## Обзор

[Kubernetes](https://en.wikipedia.org/wiki/Kubernetes) aka K8S = open <u>container-orchestration system</u> for deployment, scaling and management automation<br>
*in 99% of cases container = Docker container

<img src="img/kubernetes.png" width=500>

В 2015 году приложения стали все больше микросервисы. Google за-опенсорсил свой инфраструктурный конструктор, который позволял им быстро конфигурировать и разворачивать распределнные приложения

В концепции Kubernetes приложение ставится сразу на кластер, одна нода выбирается как master, остальные - workers<br>
Pod = обертка над контейнером (чаще всего одним, но возможно нескольких)<br>
kubelet - локальный сервис на Worker-ах, отвечающй за кправление Pod-ами<br>
kube-proxy - локальный сервис на Worker-ах, отвечающий за сетевую маршрутизацию<br>
API Server - точка входа для обработки пользовательских коман<br>
Scheduler - отвечает за выделение ресурсов<br>
Controller Manager

Why?<br>
compatibility matrix<br>
different app components require different dependency versions!<br>
new developers have to configure a lot of everything

Solution<br>
containerize each component



## Варианты запуска

__1. Локально__<br>
Есть утилита minikube, которая создает контейнер с 1-node кластером kubernetes. Идеально, чтобы изучить синтаксисё

__2. AWS__<br>
На AWS сервис называется EKS (Elastic Kubernetes Service). Можно создавать в GUI, а также есть management утилита eksctl

создать кластер по умолчанию<br>
`eksctl create cluster`

создать кластер из конфига
`eksctl create cluster -f <config>`

Типовой EKS-конфиг с опиманием кластера содержит блоки:<br>nodeGroups сколько и каких нод нам нужно<br>vpc - сетевыенастройки<br>meta - своя метаинформация

В момент запуска генерируется конфиг для Kubernetes .kube/config. Благодаря нему команда kubectl знает местоположение кластера в облаке<br>
`kubectl get-nodes`

__3. Google Cloud__<br>
```
# Google требует сначала явно включить опцию
gcloud services enable continaer

# создать кластер по-умолчанию
gcloud container clusters create

# создать кластер с параметрами
gcloud container clusters create --num_nodes=2

# получить конфиг для подключения через kubectl
gcloud container clusters get_credentials
```

__4. Labs__<br>
Есть сервис-песочница. Максимум 5 nodes, 4 часа

__How it solves the problem__

OS = kernel + software<br>
For example Ubuntu, openSUSE, CentOS all share the same kernel (Linux), differ in software<br>
Docker = software

VMs vs Docker<br>
Docker shares the same OS<br>
Docker is better in Utilization and Size => Speed<br>
VMs can run different OS

Where Docker applications reside<br>
Docker Hub = public store, everyone can contribute<br>
docker run <name> - runs a containered application<br>
Image = packaged docker application

Before:<br>
Developer prepares Application + deployment instruction<br>
Devops fails to deploy and they both figure out why

Now:<br>
Developer prepares Docker container<br>
It runs smoothly everywhere

Orchestration = auto scaling in the cluster
Kubernetes = container orchestration tool

Альтернативные решения:
- Docker Swarm
- Apache Mesos

Nodes have to be physical (why?)


## Архитекутура

Master - nodes

Services:
- API server - gateway для кластера
- etcd - configuration service (zookeeper)
- scheduler - distributes work
- controller - monitors execution
- container runtime - software (in most cases docker containers)
- kubelet - gateway of the node

#### Основная команда kubectl

Команда kubectl - это командная строка для разверттывания<br>
`kubectl run <name>` - deploy on server

Примеры развернтывания отдельных POD-ов<br>
```
# развернуть новый pod из образа nginx
kubectl run nginx_pod --image=nginx

# развернуть новый pod из манифеста
kubectl create -f pod-def.yml

# проапдейтить под из манифеста
kubectl apply -f pod-def.yml

# либо можно на ходу отредактировать (the same as get / edit / apply)
kubectl edit pod nginx

# при запуске сконвертировать аргументы команды в YAML-файл
kubectl run nginx --image=nginx -o yaml > pod.yml

# то же самое, но без запуска
kubectl run nginx --image=nginx --dry-run=client -o yaml > pod.yml
```

На практике поды обычно разворачивают не через командную строку, а через конфиг

Пример минимального конфиг файла для развертывания Pod-а
```
apiVersion: v1
kind: Pod
metadata:
    # tags and labels
spec:
    containers:
        - name:
        - image:
```

В блоке containers можно прописать один image, можно несколько. На практике один под = один контейнер

Удобно писать через VSC (или PyCharm) с установленным Kubernetes plugin-ом, он проверяет, делает auto-complete и форматирует

Удалить запущенный POD<br>
`kubectl delete pod <name>`

Удалить все развернутые POD-ы<br>
`kubectl delete --all pods`

Описать POD<br>
`kubectl describe pods`

Запустить команду внутри контейнера<br>
`kubectl exec`

Запустить shell веутри контейнера<br>
`kubectl exec -it`

Показать логи по POD-у<br>
`kubectl logs`

Замапить внутренний порт на внешний<br>
`kubectl port-forward`

Развренуть POD из manifest файла<br>
`kubectl apply -f <config>`

POD Replication
Since Kunernetes is an orchestration tool, it must support scaling, resilience and load balancing.
ReplicaSet controller maintains the exact amount of POD instances on the cluster.

In Kubernetes there are two things:
RepliactionController (an old one)
ReplicaSet (a new one)
The only difference - ReplicaSet can control those PODs that were already created without any replication - it has a selector functionality.

Replicasets are created from configuration file.
kubectl create replicaset -f rs.yml

```
spec:
    template: full POD description
    replicas: 3
    selector:
```

Template is required even when using a selector for old PODs. Because RS needs to know how to start new instances.

Replication starts immediately
If there are more active PODs than defined, they are terminating. 
If there are less PODs, new ones are starting

To print all POD instances (including those from ReplicaSet)
`kubectl get pods`

To print all ReplicaSets
`kubectl get replicaset`

To get more details:
`kubectl describe replicaset <rs-name>`
`kubectl describe pod <pod-name>`

#### ReplicaSet

#### Deployment
Объект Deployment - более высокоуровневая обертка над POD-ами set of replicasets

Она абстрагирует от низкоуровенвой поддержки на уровне подов для обеспечения scaling / availability.

Создать и развернуть новый deployment из Docker-образа<br>
`create deployment --image <image>`

Посмотреть все развернутые deployment-ы<br>
`get deployment`

По умолчанию создается один POD из образа <image>

Главная фишка - входящие в deployment POD-ы можно обновлять на ходу

Как создать Deployment из конфигурационного файла. Выглядит аналогично ReplicaSet
```
apiVersion
kind: Deployment
metadata:
spec:
  replicas:
  template:
  selector:
```

Вывести все ресурсы
`kubectl get all`

Отредактировать на ходу:
`kubectl apply -f file`<br>
`kubectl set image deployment nginx`<br>
`kubectl edit deployment mydep`

Use-cases for update:<br>
upgrading a version of an app (using another Docker image)

POD scaling does not trigger Rollout

Две стратегии:
- rolling update (default) = reset PODs one-by-one
- recreate = reset all PODs at once

Стратегия выбирается в конфиг файле:
```
spec:
  strategy:
    type: Recreate | Rollout
```

How to manage rollout (deployment)?
`kubectl rollout status deployment mydep`
`kubectl rollout history deployment mydep`
`kubectl rollout undo deployment mydep`

How to change replication for active ReplicaSets?

```
# By manually editing RS definition
kubectl edit replicaset rs

# By using replace
kubectl replace replicaset

# By using scale command
kubectl scale replicas 3 replicacset rs
```

Развернуть deployment с 4 репликами<br>
`kubectl scale deployment --replicas=4`

Настроить автомасштабируемый Deployment<br>
`kubectl autoscale deployment --min=4 --max=8 --cpu-percent=80`

HPA = Horizontal PodAutoscaler

Metrics - метрики, которые должен мониторить AutoScaler и разворачить свормсит новые инстансы

set image
rollout undo
rollout history
rollout statuses






#### Пример microservice приложения

It's a standard Docker example.

It consists of 5 components:<br>
- Voting Web App<br>
written on Python Flask
- Result Web App<br>
written on Node.JS
- Redis Backend<br>
for storing incoming votes
- Postgres Backend<br>
for storing resulting stats
- Worker process<br>that transfers vote from Redis to Postgres 

Web app face public network => there must be NodePort defined

There should be 4 services:
- Public services for Voting and Results web applications
- NodePort / LoadBalancer
- Private services for reading and writing Redis and Postgres
- ClusterIP

Total 9 definition files

It's better to wrap each POD into deployment. In that chase you can scale easily
kubectl scale deployment --replicas 3

By default deployment creates a ReplicaSet

To get several resources:
`kubectl get pods, svc`

Container ports
When defining PODs you also configure ports that gonna be used: 
```
spec:
    container:
        ports: 
            containerPort: 80
```

Environament variables
You can also configure environment variables (for example, for user / password) 

```
spec:
    container:
        env: 
name: 
            	value: 
name: 
value: 
```

Managed K8s Clusters

There are 3 options:
- GCP
- AWS
- Azure

Setting up cluster in AWS is the most involved process

Schema is the following<br>
you create a cluster<br>
configurate kubectl to manage the created cluster<br>
git clone all your POD definitions and deploy them on the cluster

In GCP and Azure you work in cloud shell<br>
In AWS you work locally

You don't SSH on the the Worker nodes<br>
You cannot access master node!

You can vIsually inspect the K8s resource

__New cluster configuration__

How to configure?
define a set of nodes (for example as VMs)
check OS and libraries prerequisites on each node
install Docker (or other container runtime) on each node
configure it to run as a service
install K8s tools on each node
kubeadm
kubelet
kubectl
initialize Master
run kubeadm init on the master node
add argument to define subnetwork for PODs 
(POD network will be added using 3rd party software)
add IP address for apiServer
in home directory add a config file
create a POD network by applying a configuration file
kubectl apply -f "..."
Join other nodes to a cluster
kubeadm join 

Commands can be run using kubectl on any node of the cluster.

__Vagrant__ = tool for VM automation
You can define all machine parameters in a file and run it.

## HeLM
__HEML__ - шаблонизатор манифестов для Kubernetes. Когда число деплойментов переваливает щза сотню управлять ими становится невозмонжно. Здесь переходят на уровень абстрактности выше и параметризуют манифесты. Вместо апдейта множества подов достаточно обновить условно один праметр и сделать apply

## Пример ML процесса на Kubernetes
1. Создаем Docker Image на базе готового образа с библиотекой transformers
2. Запускаем подгрузку нужной нам модели
3. В качестве основной команды делаем serve - теперь модель доступна по API (используется fastapi + uvicorn)

### Сервисы (Services)
Поды по своей природе динамичные - они могут рандомно пересоздаваться и когда они создаются, у них меняется IP адрес => нужен некоторый мапер, который будет:
1. предоставлять внешний ip-дарес
2. мапить его на внутренние адреса подов + случайную маршрутизацию

В Kubernetes такой ресурс называется Service. Это процесс, разворачивающийся на одной их нод и испольщзующийся для сетевой маршрутизации запросов

В командной строке сервис создается командой expose (открыть порт):<br>
```
kubectl expose deployment 
    --type=<type> 
    --port=<outer_port>:<pod_port>
```

Посмотреть развернутые сервисы<br>
`kubectl get svc`

### Типы сервисов
В Kubernetes есть несколько типа сервисов:
- ClusterIP - обеспечивающий доступ внтури кластера
- NodePort - обеспечивающий внешний доступ
- LoadBalancer - создается внешний сервис с балансировщиком в облаке (AWS / GCP)
- ExternalService - внешний сервис

<img src="img/services.png" width=500>

Как это делается через конфигурационный файл:
```
# создаем Сервис
kind: Service

# на какие поды распространяется
selector:
  - label:

# параметры сервиса
spec:
  - type:

# какие порты открывать
ports:
  - name:
  - name:
```
### Ingress Controllers
В архитекутре систем термин Ingress - это собирательное название некоторого gateway сервиса для всего входящего внешнего траффика

Если нужна более сложная маршрутизация, её можно сделать отдельным приложением. В зависимости от того, какой dns в запросе, перенеаправлять на нужный backend. Такое приложение называется Ingress Controller

<img src="img/ingress.png" width=500>

Есть много реализаций от разных проихводителей - зачем?

### Namespace
Неймспейсы - способ раздеять пространства имен ресурсов. В рамках одного неймспейса ресурсы должны быть уникальными, 

Полезно, когда используются разные среды